# Triple-Hybrid RAG — Full Pipeline
논문: *Performance Optimization Study of Hybrid RAG Engine Integrating Multi-Source Knowledge*

**실행 순서**: 위에서 아래로 순서대로 셀 실행

In [ ]:
# ── Step 0: 설치 ──────────────────────────────────────
!pip install -q langchain>=0.1.0 langchain-openai>=0.0.5 langchain-community
!pip install -q faiss-cpu numpy openai
!pip install -q owlready2 neo4j
!pip install -q streamlit plotly koreanize-matplotlib pandas
print('✅ 설치 완료')

In [ ]:
# ── Step 1: GitHub 클론 ───────────────────────────────
!git clone https://github.com/sdw1621/hybrid-rag-comparsion.git
%cd hybrid-rag-comparsion
import sys; sys.path.insert(0, '.')
print('✅ 레포 클론 완료')

In [ ]:
# ── Step 2: API 키 설정 ───────────────────────────────
import os
from google.colab import userdata
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('✅ API 키 로드 완료 (Colab Secrets)')
except:
    os.environ['OPENAI_API_KEY'] = input('OpenAI API Key: ')

In [ ]:
# ── Step 3: QueryAnalyzer + DWA 검증 ─────────────────
from src import QueryAnalyzer, DWA

analyzer = QueryAnalyzer()
dwa      = DWA(lambda_=0.3)

test_queries = [
    '김철수 교수의 연구 분야는 무엇인가요?',
    '김철수 교수가 소속된 학과의 40세 이하 교수 목록은?',
    'AI 프로젝트 참여 교수 중 컴퓨터공학과 소속은 누구인가요?',
]

for q in test_queries:
    intent = analyzer.analyze(q)
    w      = dwa.compute(intent)
    print(f'Q: {q}')
    print(f'  유형={intent.query_type} | c_e={intent.c_e:.2f} c_r={intent.c_r:.2f} c_c={intent.c_c:.2f}')
    print(f'  {w}')
    print()

In [ ]:
# ── Step 4: TripleHybridRAG 빌드 ──────────────────────
from src import TripleHybridRAG

rag = TripleHybridRAG(lambda_=0.3, top_k=3)
rag.load_university_sample()
rag.build()

In [ ]:
# ── Step 5: 단일 질의 테스트 ──────────────────────────
result = rag.query('김철수 교수가 담당하는 과목은 무엇인가요?')

print(f'답변: {result.answer}')
print(f'응답시간: {result.elapsed:.3f}s')
print(f'가중치: {result.weights}')
print(f'질의유형: {result.intent.query_type}')
print(f'\nVector 컨텍스트:')
for c in result.vector_contexts: print(f'  - {c[:80]}')
print(f'\nGraph 컨텍스트:')
for c in result.graph_contexts:  print(f'  - {c}')
print(f'\nOntology 컨텍스트:')
for c in result.onto_contexts:   print(f'  - {c}')

In [ ]:
# ── Step 6: Gold QA 데이터셋 생성 (5,000쌍) ──────────
from data.dataset_generator import save_dataset

dataset = save_dataset('data/gold_qa_5000.json', total=5000)
print(f'총 {len(dataset)}개 생성 완료')
print(f'\n샘플 3개:')
for d in dataset[:3]:
    print(f'  [{d["type"]}] Q: {d["query"]} → A: {d["answer"]}')

In [ ]:
# ── Step 7: 평가 실행 (샘플 100개, 빠른 검증) ──────────
# 전체 5,000쌍 평가는 API 비용이 크므로 100개 샘플로 경향성 확인
import json, random
from src import Evaluator

with open('data/gold_qa_5000.json', encoding='utf-8') as f:
    full_ds = json.load(f)

print(f'전체 Gold QA: {len(full_ds)}쌍 (Simple {sum(1 for d in full_ds if d["type"]=="simple")}개 / '
      f'Multi-hop {sum(1 for d in full_ds if d["type"]=="multi_hop")}개 / '
      f'Conditional {sum(1 for d in full_ds if d["type"]=="conditional")}개)')

random.seed(42)
sample_ds = random.sample(full_ds, 100)

evaluator = Evaluator()
summary   = evaluator.evaluate_dataset(rag, sample_ds, runs=1, verbose=False)

print('\n=== 평가 결과 (Table 13 형식) ===')
for metric, vals in summary.items():
    if metric != 'by_type':
        print(f'  {metric:15}: {vals["mean"]:.4f} ± {vals["std"]:.4f}')
print('\n질의 유형별 EM:')
for t, vals in summary['by_type'].items():
    print(f'  {t:15}: {vals["mean"]:.4f} ± {vals["std"]:.4f}')

In [ ]:
# ── Step 8: Ablation Study ────────────────────────────
from src import AblationStudy

def rag_factory():
    r = TripleHybridRAG(lambda_=0.3, top_k=3)
    r.load_university_sample()
    r.build()
    return r

ablation = AblationStudy(rag_factory, sample_ds)
abl_results = ablation.run(sample_size=30, runs=1)

In [ ]:
# ── Step 9: Streamlit 앱 실행 (Colab 터널링) ──────────
!pip install -q pyngrok
from pyngrok import ngrok
import subprocess, time

proc = subprocess.Popen(
    ['streamlit', 'run', 'streamlit_app/app.py',
     '--server.port', '8501', '--server.headless', 'true'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)
public_url = ngrok.connect(8501)
print(f'🚀 Streamlit 앱 URL: {public_url}')